In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import confusion_matrix
from sklearn.metrics import recall_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.metrics import f1_score
from sklearn.metrics import fbeta_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.calibration import LabelEncoder
from sklearn.discriminant_analysis import StandardScaler
from xgboost import XGBClassifier


import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings("ignore")

In [101]:
train = pd.read_csv("datasets/training_data.csv", encoding='utf-8', encoding_errors='ignore')
test = pd.read_csv("datasets/test_data.csv", encoding='utf-8', encoding_errors='ignore')

In [102]:
train = train.drop(columns=["AVERAGE_PRECIPITATION", "AVERAGE_RAIN"], errors='ignore')
test = test.drop(columns=["AVERAGE_PRECIPITATION", "AVERAGE_RAIN"], errors='ignore')

train["record_date"] = pd.to_datetime(train["record_date"])
test["record_date"] = pd.to_datetime(test["record_date"])

In [ ]:

# CONVERTER COLUNAS NUMÉRICAS
num_cols = [
    "AVERAGE_FREE_FLOW_SPEED", "AVERAGE_TIME_DIFF", "AVERAGE_FREE_FLOW_TIME",
    "AVERAGE_TEMPERATURE", "AVERAGE_ATMOSP_PRESSURE", "AVERAGE_HUMIDITY",
    "AVERAGE_WIND_SPEED", "AVERAGE_CLOUDINESS", "AVERAGE_SPEED_DIFF",
    "LUMINOSITY"
]

for df in [train, test]:
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")


# Extrair partes do tempo
for df in [train, test]:
    df["hour"] = df["record_date"].dt.hour
    df["dayofweek"] = df["record_date"].dt.dayofweek
    df["month"] = df["record_date"].dt.month
    df["is_weekend"] = df["dayofweek"].isin([5,6]).astype(int)
    
    # Codificação cíclica (hora e mês)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"]/24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"]/24)
    df["month_sin"] = np.sin(2 * np.pi * df["month"]/12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"]/12)
    
    #HORAS DE PONTA E PARTE DO DIA
    df["is_rush_hour"] = df["hour"].isin([7,8,9,17,18,19]).astype(int)
    def part_of_day(h):
        if 6 <= h < 12: return "morning"
        elif 12 <= h < 18: return "afternoon"
        elif 18 <= h < 24: return "evening"
        else: return "night"
    df["part_of_day"] = df["hour"].apply(part_of_day)

# One-hot encoding da parte do dia
train = pd.get_dummies(train, columns=["part_of_day"], drop_first=True)
test = pd.get_dummies(test, columns=["part_of_day"], drop_first=True)


# INTERAÇÕES ENTRE VARIÁVEIS
for df in [train, test]:
    # Só aplica ao train que o test nao tem o average def speed
    train["efficiency_ratio"] = train["AVERAGE_FREE_FLOW_SPEED"] / (train["AVERAGE_TIME_DIFF"] + 1e-6)
    train["delay_ratio"] = train["AVERAGE_TIME_DIFF"] / (train["AVERAGE_FREE_FLOW_TIME"] + 1e-6)
    train["speed_time_ratio"] = train["AVERAGE_SPEED_DIFF"] / (train["AVERAGE_TIME_DIFF"] + 1e-6)

    # Para o test (sem o target)
    test["efficiency_ratio"] = test["AVERAGE_FREE_FLOW_SPEED"] / (test["AVERAGE_TIME_DIFF"] + 1e-6)
    test["delay_ratio"] = test["AVERAGE_TIME_DIFF"] / (test["AVERAGE_FREE_FLOW_TIME"] + 1e-6)

    # Clima x Luminosidade
    df["low_light"] = (df["LUMINOSITY"] < df["LUMINOSITY"].quantile(0.2)).astype(int)
    df["cloud_luminosity_ratio"] = df["AVERAGE_CLOUDINESS"] / (df["LUMINOSITY"] + 1e-6)
    df["wind_cloud_interaction"] = df["AVERAGE_WIND_SPEED"] * df["AVERAGE_CLOUDINESS"]



In [ ]:
#xgboost SO TRABALHA COM VALORES NUMÉRICOS
le = LabelEncoder()
train["AVERAGE_SPEED_DIFF"] = le.fit_transform(train["AVERAGE_SPEED_DIFF"])
y = train["AVERAGE_SPEED_DIFF"]

X = train.drop(columns=["AVERAGE_SPEED_DIFF", "record_date"])
X_test = test.drop(columns=["record_date"], errors='ignore')

X = pd.get_dummies(X, columns=["city_name", "LUMINOSITY", "AVERAGE_CLOUDINESS"], drop_first=True)
X_test = pd.get_dummies(X_test, columns=["city_name", "LUMINOSITY", "AVERAGE_CLOUDINESS"], drop_first=True)

X_test = X_test.reindex(columns=X.columns, fill_value=0)

#esta a converter em colunas numéricas


In [ ]:

# NORMALIZAÇÃO DOS DADOS

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)



In [106]:
from sklearn.metrics import classification_report
from sklearn.utils import compute_sample_weight


model = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    objective='multi:softmax',
    num_class=len(np.unique(y)),
    eval_metric='mlogloss'
)

X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Calcular pesos automáticos (quanto menor a classe, maior o peso)
weights = compute_sample_weight(class_weight='balanced', y=y_train)

# Treinar o modelo com os pesos
model.fit(X_train, y_train, sample_weight=weights)
# Avaliar
y_pred = model.predict(X_val)
print("Accuracy:", accuracy_score(y_val, y_pred))
print("\nMatriz de confusão:\n", confusion_matrix(y_val, y_pred))
print("\nRelatório de classificação:\n", classification_report(y_val, y_pred))

Accuracy: 1.0

Matriz de confusão:
 [[1363]]

Relatório de classificação:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      1363

    accuracy                           1.00      1363
   macro avg       1.00      1.00      1.00      1363
weighted avg       1.00      1.00      1.00      1363



In [107]:
train["AVERAGE_SPEED_DIFF"].value_counts()



AVERAGE_SPEED_DIFF
0    6812
Name: count, dtype: int64

In [108]:
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1363

    accuracy                           1.00      1363
   macro avg       1.00      1.00      1.00      1363
weighted avg       1.00      1.00      1.00      1363

